In [ ]:
SincConv


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SincConv(nn.Module):
    def __init__(self, out_channels=16, kernel_size=101, sr=16000):
        super().__init__()

        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.sr = sr

        # learnable cutoff frequencies
        self.low_hz = nn.Parameter(torch.linspace(0, 3000, out_channels))
        self.band_hz = nn.Parameter(torch.ones(out_channels) * 1000)

    def forward(self, x):
        device = x.device

        t = torch.linspace(
            -(self.kernel_size//2),
            self.kernel_size//2,
            steps=self.kernel_size
        ).to(device)

        filters = []

        for i in range(self.out_channels):
            f1 = torch.abs(self.low_hz[i]) + 1
            f2 = f1 + torch.abs(self.band_hz[i])

            sinc1 = torch.sin(2 * torch.pi * f1 * t / self.sr) / (t + 1e-6)
            sinc2 = torch.sin(2 * torch.pi * f2 * t / self.sr) / (t + 1e-6)

            band_pass = sinc2 - sinc1
            filters.append(band_pass)

        filters = torch.stack(filters).unsqueeze(1)  # (C,1,K)

        return F.conv1d(x, filters, padding=self.kernel_size//2)

In [ ]:
Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(16, 32, kernel_size=3)
        self.bn1   = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.bn2   = nn.BatchNorm1d(64)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = torch.relu(x)

        return x

In [ ]:
encoder = Encoder()
sinc = SincConv()

In [ ]:
Path-1

In [ ]:
KAN-GAL (attention)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)

    def forward(self, x):
        Q = self.linear(x)
        K = self.linear(x)
        V = self.linear(x)

        attn = torch.matmul(Q, K.transpose(1, 2)) / (Q.size(-1) ** 0.5)
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, V)
        return out

In [ ]:
class SpatialAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)

    def forward(self, x):
        # x: (N, 64, T)

        x = x.permute(0, 2, 1)   # (N, T, 64)

        Q = self.linear(x)
        K = self.linear(x)
        V = self.linear(x)

        attn = torch.matmul(Q, K.transpose(1, 2))
        attn = F.softmax(attn, dim=-1)

        out = torch.matmul(attn, V)

        return out.permute(0, 2, 1)  # back to (N,64,T)

In [ ]:
pooling

In [ ]:
class TemporalPool(nn.Module):
    def __init__(self, k=50):
        super().__init__()
        self.k = k

    def forward(self, x):
        # x: (N, T, 64)

        scores = x.mean(dim=2)   # (N, T)
        topk = torch.topk(scores, self.k, dim=1).indices

        # gather top nodes
        out = torch.gather(
            x,
            1,
            topk.unsqueeze(-1).expand(-1, -1, x.size(2))
        )

        return out   # (N, k, 64)

In [ ]:
class SpatialPool(nn.Module):
    def __init__(self, k=32):
        super().__init__()
        self.k = k

    def forward(self, x):
        # x: (N, 64, T)

        scores = x.mean(dim=2)   # (N, 64)
        topk = torch.topk(scores, self.k, dim=1).indices

        out = torch.gather(
            x,
            1,
            topk.unsqueeze(-1).expand(-1, -1, x.size(2))
        )

        return out   # (N, k, T)

In [ ]:
KAN GAL (2nd Path)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KANLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return torch.tanh(self.linear(x))   # placeholder KAN


class KAN_GAL(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()

        self.kan = KANLayer(in_dim, out_dim)
        self.attn = nn.Linear(out_dim * 2, 1)

    def forward(self, h):
        # h: [N, F]

        # 1. KAN transform
        h = self.kan(h)   # [N, F]

        N = h.size(0)

        # 2. pairwise attention
        h_i = h.unsqueeze(1).expand(-1, N, -1)  # [N, N, F]
        h_j = h.unsqueeze(0).expand(N, -1, -1)  # [N, N, F]

        # 3. concat
        attn_input = torch.cat([h_i, h_j], dim=-1)  # [N, N, 2F]

        # 4. attention score
        e = self.attn(attn_input).squeeze(-1)  # [N, N]

        # 5. normalize
        alpha = F.softmax(e, dim=1)

        # 6. aggregate
        h_out = torch.matmul(alpha, h)   # [N, F]

        return h_out

In [ ]:
Graph Pooling

In [ ]:
import torch
import torch.nn as nn

class GraphPool(nn.Module):
    def __init__(self, in_dim, k=50):
        super().__init__()
        self.score_layer = nn.Linear(in_dim, 1)
        self.k = k

    def forward(self, x):
        # x: [N, F]

        scores = self.score_layer(x).squeeze()   # [N]

        # top-k nodes
        k = min(self.k, x.size(0))
        idx = torch.topk(scores, k=k).indices

        h = x[idx]   # [K, F]

        return h

In [ ]:
KAN-HS-GAL

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class KANLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        # simple KAN approximation (replace later with real KAN)
        return torch.tanh(self.linear(x))


class KAN_HS_GAL(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()

        # KAN transform
        self.kan = KANLayer(in_dim, out_dim)

        # attention weights
        self.attn = nn.Linear(out_dim * 2, 1)

    def forward(self, h):
        # N <= 200   # borderline
        # h: [N, F]

        # 1. KAN transform
        h = self.kan(h)   # [N, F]

        N = h.size(0)

        # 2. pairwise attention (IMPORTANT: costly if large N)
        h_i = h.unsqueeze(1).repeat(1, N, 1)  # [N, N, F]
        h_j = h.unsqueeze(0).repeat(N, 1, 1)  # [N, N, F]

        # concat
        attn_input = torch.cat([h_i, h_j], dim=-1)  # [N, N, 2F]

        # 3. attention score
        e = self.attn(attn_input).squeeze(-1)  # [N, N]

        # 4. normalize
        alpha = F.softmax(e, dim=1)  # row-wise

        # 5. aggregate
        h_out = torch.matmul(alpha, h)  # [N, F]

        return h_out

In [ ]:
Full Model (clean)

In [ ]:
class FullModel(nn.Module):
    def __init__(self, feat_dim=64, k=100):
        super().__init__()

        self.kan_gal = KAN_GAL(feat_dim, feat_dim)
        self.graph_pool = GraphPool(feat_dim, k)
        self.kan_hs_gal = KAN_HS_GAL(feat_dim, feat_dim)

        # ADD DROPOUT
        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(feat_dim * 4, 2)

    def forward(self, x):
        B = x.size(0)
        outputs = []

        for i in range(B):
            xi = x[i]   # [F, T]

            # --- graph path ---
            x_graph = xi.permute(1, 0)   # [T, F]

            h = self.kan_gal(x_graph)
            h = self.graph_pool(h)
            h = self.kan_hs_gal(h)

            h_max = torch.max(h, dim=0).values
            h_mean = torch.mean(h, dim=0)

            # --- pooling path ---
            h_pool = torch.mean(xi, dim=1)

            L = torch.cat([h_max, h_mean, h_pool, h_pool], dim=0)

            outputs.append(L)

        L = torch.stack(outputs)   # [B, 4F]

        # APPLY DROPOUT HERE
        L = self.dropout(L)

        out = self.classifier(L)

        return out